<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">
<br>
<h1 style="font-family:verdana; font-weight: bold; color: #5A4636; text-align: center;">
Intelligo AI Course Advisor<br>
<span style="font-size: 22px;">End-to-End Notebook</span>
</h1>

<p style="text-align:center; font-size:16px; color:#6F5C4F;">
Tim <b>psi-1</b>
</p>

<p style="text-align:center; font-size:15px; color:#6F5C4F; line-height:1.7;">
Ketua - <b>Aufar Kusuma</b><br>
Anggota - <b>Mochamad Fachri Alfaridzi</b><br>
Anggota - <b>Kurt Mikhael Purba</b>
</p>




<hr style="border: 2px solid #D32F2F; margin-top: 10px; margin-bottom: 10px;">
<div style="background-color: #FFEBEE; border-left: 6px solid #D32F2F; padding: 16px; margin: 8px 0; border-radius: 4px;">
<h3 style="margin-top: 0; color: #C62828;">[PERINGATAN] Penting untuk Reproduktibilitas — WAJIB CPU, JANGAN pakai GPU</h3>
<p style="margin-bottom: 8px; color: #4A148C; font-size: 15px; line-height: 1.6;">
<strong>Sangat disarankan / diharuskan menjalankan pipeline ini di Kaggle Notebook dengan CPU saja (non-GPU, Accelerator = None).</strong>
</p>
<p style="margin-bottom: 8px; color: #4A148C; font-size: 15px; line-height: 1.6;">
Alasan: pipeline ini memakai threading deterministik (thread GBDT 8, thread BLAS 1) dan <code>threadpoolctl</code> untuk mengunci pool BLAS agar bit terakhir hasil identik antar mesin. GPU memperkenalkan non-determinisme melalui operasi BLAS/CUDA paralel yang tidak bisa dikunci penuh, sehingga submission hasil run GPU <strong>tidak akan identik bit-per-bit</strong> dengan submission referensi meski seed sama — agar reproducible, JANGAN aktifkan GPU.
</p>
<p style="margin-bottom: 0; color: #4A148C; font-size: 15px; line-height: 1.6;">
Jika dijalankan di Kaggle: <em>Settings -> Accelerator -> None (CPU)</em>. Jika di lokal; pastikan <code>CUDA_VISIBLE_DEVICES=""</code> sebelum import numpy/LightGBM/XGBoost.
</p>
</div>
<hr style="border: 2px solid #D32F2F; margin-top: 10px;">

# Daftar Isi
1. [Pendahuluan](#1)
2. [Problem Statement dan Ketentuan Lomba](#2)
3. [Dataset Overview](#3)
4. [Initialization](#4)
5. [Exploratory Data Analysis (EDA)](#5)
6. [Preprocessing](#6)
7. [Modelling](#7)
8. [Training dan Submission](#8)
9. [Bobot Tersimpan dan Shortcut](#9)
10. [Reproducibility](#10)
11. [Post-Insight Analysis](#11)

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Pendahuluan <a name="1"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Intelligo adalah ekosistem pembelajaran modern tempat setiap siswa memiliki latar, pemahaman, minat, dan tujuan karier yang berbeda. Seorang pemula membutuhkan fondasi pengolahan data, calon data scientist membutuhkan materi Machine Learning, dan profesional membutuhkan materi lanjutan seperti MLOps.

Sistem rekomendasi konvensional cenderung memberi rekomendasi populer atau pola umum tanpa memahami kebutuhan setiap individu. Akibatnya materi yang muncul belum tentu cocok dengan kemampuan, tujuan, maupun tahap karier pengguna.

Notebook ini adalah solusi final Tim **psi-1** untuk kompetisi **MineToday IT Today 2026**. Tujuannya menjawab satu pertanyaan untuk setiap pengguna.

“Modul apa yang paling relevan untuk dipelajari oleh pengguna tersebut”

Target akhirnya adalah peringkat 5 modul teratas per pengguna yang diurut dari paling relevan dan dinilai dengan NDCG@5.

**Batasan yang dijaga** di notebook ini
- Satu notebook mandiri dari CSV mentah sampai file submission
- Affinity teks ke modul selalu dihitung nested out-of-fold agar label tidak bocor ke fitur
- Pemilihan komponen dan bobot hanya berdasarkan validasi internal 5-fold

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Problem Statement dan Ketentuan Lomba <a name="2"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

## Problem Statement

Tugasnya adalah **learning to rank** untuk 17 modul Intelligo. Setiap pengguna latih memiliki skor relevance per modul dan setiap pengguna uji harus diprediksi skornya dengan struktur yang sama. Output akhirnya adalah peringkat 5 modul teratas per pengguna yang terurut dari paling relevan.

Skala relevance berjalan dari 0 untuk tidak relevan sampai 1 untuk paling relevan. Setiap pengguna memiliki 4 sampai 6 modul yang nilainya bukan nol sehingga peringkat yang benar harus menempatkan modul 1.0 di posisi teratas.

---

## Ketentuan Lomba dan Evaluasi

- Tugasnya adalah peringkat top-5 modul per pengguna dalam format `sample_submission.csv`
- Metrik utamanya adalah **NDCG@5** yang dirata-rata per pengguna
- Setiap pengguna uji wajib memiliki tepat 5 modul teratas yang terurut

$$\text{NDCG@5} = \frac{\text{DCG@5}}{\text{IDCG@5}} \quad\text{dengan gain linear dari relevance}$$

Metrik ini dipilih karena urutan sangat menentukan. Modul yang relevan tetapi jatuh ke posisi bawah hampir tidak menyumbang skor, jadi seluruh strategi model di notebook ini dioptimalkan untuk akurasi posisi teratas.

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Dataset Overview <a name="3"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

## Sumber Dataset

Enam CSV kompetisi dibaca dari `data/raw` dan otomatis berjalan juga di Kaggle bila folder input tersedia.

---

## Ringkasan Dataset

| File | Baris | Isi |
|---|---|---|
| `train_relevance.csv` | 4000 | `user_id` dan relevance `M_001` sampai `M_017` |
| `test.csv` | 1000 | `user_id` yang harus diprediksi |
| `chat_history.csv` | 14573 | `user_id`, `chat_id`, `timestamp`, `user_chat_text` |
| `user_assessments.csv` | 5000 | `user_id` dan `assessment_result` berisi JSON 10 soal |
| `modules_catalog.csv` | 17 | `module_id`, `module_name`, `description_and_syllabus`, `prerequisite_level` |
| `sample_submission.csv` | 1000 | format dan urutan submission yang wajib diikuti |

Data latih berisi `U_0001` sampai `U_4000` dan data uji berisi `U_4001` sampai `U_5000` tanpa tumpang tindih. Urutan `user_id` di data uji sama persis dengan `sample_submission.csv`.

---

## Struktur Direktori

```text
data/raw/
submissions/
outputs/eda/
outputs/oof/
outputs/bobot/
```

Seluruh file submission, bobot, dan artefak OOF ditulis ke folder di atas agar alur data dapat diaudit ujung ke ujung.

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Initialization <a name="4"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Bagian ini menyiapkan tiga fondasi sebelum pipeline berjalan.

1. **Penyiapan lingkungan** memastikan thread GBDT dan thread BLAS dikunci terpisah agar bit terakhir hasil tetap sama antarmesin
2. **Penyiapan reproduksibilitas** menetapkan seed dan konstanta global seperti `SEED`, `N`, dan `NTHREAD`
3. **Penyiapan direktori** menemukan folder enam CSV kompetisi baik di lokal maupun di Kaggle

In [1]:
# kunci thread BLAS ke 1 sebelum numpy diimpor agar bit terakhir stabil antar mesin
import os

for _v in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "OMP_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"

import json, re, time, warnings, collections, hashlib, platform
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost
import sklearn
import threadpoolctl
from catboost import CatBoostRegressor
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, StratifiedKFold

# simpan objeknya di variabel agar setting tidak hilang
BLAS_LOCK  = threadpoolctl.threadpool_limits(limits=1, user_api="blas")
BLAS_POOLS = [(p["internal_api"], p["num_threads"]) for p in threadpoolctl.threadpool_info()
              if p["user_api"] == "blas"]      # pool "openmp" milik GBDT, diatur NTHREAD
assert all(n == 1 for _, n in BLAS_POOLS), f"pool BLAS belum terkunci ke 1: {BLAS_POOLS}"

warnings.filterwarnings("ignore")
pd.set_option("display.width", 150)

_kandidat = [Path("../data/raw"), Path("data/raw")]
if Path("/kaggle/input").exists():                       # jalan juga di Kaggle
    _kandidat += [q.parent for q in Path("/kaggle/input").rglob("train_relevance.csv")]
RAW = next((q for q in _kandidat if (q / "train_relevance.csv").exists()), None)
assert RAW is not None, ("folder enam CSV kompetisi tidak ketemu; isi /kaggle/input = "
                         + str(list(Path("/kaggle/input").rglob("*"))[:40]))
SEED, N, NTHREAD = 42, 17, 8
MODULES  = [f"M_{i:03d}" for i in range(1, 18)]
N_USERS  = 5000
W_FACTOR = 0.127        
np.random.seed(SEED)


def ndcg5(y_true, y_score, k=5):
    idx  = np.argsort(-y_score, axis=1, kind="stable")[:, :k]
    disc = 1.0 / np.log2(np.arange(2, k + 2))
    dcg  = (np.take_along_axis(y_true, idx, axis=1) * disc).sum(1)
    idcg = ((-np.sort(-y_true, axis=1)[:, :k]) * disc).sum(1)
    return float(np.mean(np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)))


def zs(a):
    return (a - a.mean(1, keepdims=True)) / (a.std(1, keepdims=True) + 1e-12)


print(f"python {platform.python_version()}  numpy {np.__version__}  pandas {pd.__version__}"
      f"  scipy {__import__('scipy').__version__}  sklearn {sklearn.__version__}")
print(f"lightgbm {lgb.__version__}  xgboost {xgb.__version__}  catboost {catboost.__version__}")
print(f"thread GBDT {NTHREAD}  |  pool BLAS {BLAS_POOLS}  |  core mesin {os.cpu_count()}")
print("siap")

python 3.11.0  numpy 2.1.3  pandas 2.3.0  scipy 1.16.0  sklearn 1.7.0
lightgbm 4.6.0  xgboost 3.1.3  catboost 1.2.8
thread GBDT 8  |  pool BLAS [('openblas', 1), ('openblas', 1)]  |  core mesin 16
siap


## Muat Data dan Kunci Urutan Pengguna

Sel berikut memuat enam CSV dan memisahkan ID pengguna latih, ID pengguna uji, dan matriks target `Y_train`. Data latih dan data uji diurut berdasarkan `user_id` sejak awal agar seluruh pengindeksan sesudahnya konsisten.

In [2]:
T0 = time.time()
train  = pd.read_csv(RAW / "train_relevance.csv").sort_values("user_id").reset_index(drop=True)
test   = pd.read_csv(RAW / "test.csv").sort_values("user_id").reset_index(drop=True)
chats  = pd.read_csv(RAW / "chat_history.csv")
assess = pd.read_csv(RAW / "user_assessments.csv")
cat    = pd.read_csv(RAW / "modules_catalog.csv")
ss     = pd.read_csv(RAW / "sample_submission.csv")

u_train, u_test = train["user_id"].to_numpy(), test["user_id"].to_numpy()
Y_train = train[MODULES].to_numpy(float)
print(f"train {train.shape}  test {test.shape}  chat {chats.shape}  assessment {assess.shape}")

train (4000, 18)  test (1000, 1)  chat (14573, 4)  assessment (5000, 2)


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Exploratory Data Analysis (EDA) <a name="5"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Sebelum membangun fitur, kita perlu memahami struktur target dan setiap sumber data. EDA ini dirancang sebagai **dasar keputusan** untuk setiap tahap sesudahnya, bukan sekadar eksplorasi.

Versi lengkap EDA beserta sebelas grafiknya tersimpan di `EDA.py` dan folder `outputs/eda`. Sel cepat di bawah merangkum angka yang sama langsung di notebook.

---

### 1. Bentuk Data dan Integritas Pembagian

Memastikan dimensi setiap file dan tidak ada kebocoran pengguna antara data latih dan data uji. Kecocokan urutan data uji ikut dicek karena satu baris yang geser membuat submission gugur. Hasilnya mengunci format sejak awal sehingga pengindeksan sesudahnya dapat mengandalkan posisi.

### 2. Distribusi Nilai Target dan Struktur Tingkat

Mengukur seberapa jarang target dan seberapa dekat nilai relevance dengan pusat tingkat. Hasilnya 96.3 persen cocok sehingga label peringkat dapat dipulihkan persis untuk model berlabel bulat.

### 3. Popularitas Modul dan Top-1

Mengukur ketimpangan popularitas modul dari 0.35 sampai 0.07 dan ketimpangan top-1 dari 28 sampai 495 pengguna. Ketimpangan ini menuntut fold terstratifikasi dan membenarkan prior popularitas modul.

### 4. Kekotoran Asesmen Mentah

Menghitung 248 angka di luar skala dan 383 angka yang dieja dari 50000 sel. Aturannya menjadi data hilang tanpa potong karena memotong berarti mengarang jawaban.

### 5. Statistik Asesmen per Pengguna

Memeriksa rata-rata, simpangan, dan kelengkapan jawaban per pengguna. Rata-rata condong ke skor rendah dan 98.5 persen pengguna menjawab penuh sehingga statistik sederhana cukup.

### 6. Cakupan dan Panjang Chat

Menghitung 150 pengguna tanpa chat serta panjang pesan yang pendek dengan median 94 karakter. Temuan ini menuntut jalur cadangan berupa prior dan mendukung fitur panjang teks.

### 7. Katalog Modul

Memetakan 12 level prasyarat sebagai fitur tabular. Panjang deskripsi berkorelasi negatif dengan popularitas sehingga deskripsi panjang menandai modul sepi.

### 8. Korelasi Antarmodul

Menguji korelasi relevance antarmodul yang ternyata hampir nol dengan nilai tertinggi 0.15. Temuan ini mendukung pembingkaian 17 regresi terpisah sebagai keluarga tersendiri.

### 9. Hubungan Fitur dengan Target

Menguji rata-rata asesmen per kuintil yang ternyata datar terhadap level global. Asesmen dipakai untuk profil level sedangkan panjang deskripsi menjadi fitur pendukung.

### 10. Ringkasan Keputusan EDA

Merangkum seluruh temuan menjadi tujuh keputusan dalam satu tabel audit. Setiap tahap sesudahnya wajib merujuk kembali ke tabel ini bila perilakunya dipertanyakan.

### 1. Bentuk Data dan Integritas Pembagian

Kita perlu memastikan dimensi setiap file benar dan tidak ada pengguna yang bocor antara data latih dan data uji. Kecocokan urutan data uji dengan contoh submission juga dicek di sini karena submission yang urutannya geser langsung gugur walau skornya benar.

In [3]:
print("bentuk train test chat assess katalog", train.shape, test.shape, chats.shape, assess.shape, cat.shape)
print("overlap user train dan test", len(set(u_train) & set(u_test)))
print("urutan test sama dengan contoh", bool((ss["user_id"].to_numpy() == u_test).all()))

bentuk train test chat assess katalog (4000, 18) (1000, 1) (14573, 4) (5000, 2) (17, 4)
overlap user train dan test 0
urutan test sama dengan contoh True


Hasilnya memastikan pembagian bersih tanpa tumpang tindih dan format submission sudah terkunci sejak awal. Dengan begitu seluruh pengindeksan pengguna sesudahnya dapat mengandalkan posisi tanpa pencarian berulang.

### 2. Distribusi Nilai Target dan Struktur Tingkat

Kita perlu tahu seberapa jarang target dan apakah nilai relevance mengelompok pada pusat tingkat. Struktur tingkat yang valid berarti label peringkat dapat dipulihkan persis untuk model yang membutuhkan label bulat.

In [4]:
Y0 = train[MODULES].to_numpy(float)
Yn = Y0[Y0 > 0]
nz0 = (Y0 > 0).sum(1)
print("nilai unik", len(np.unique(Y0)), "persen nol", round(float((Y0 == 0).mean()) * 100, 1))
print("bukan nol per pengguna", int(nz0.min()), int(np.median(nz0)), int(nz0.max()))
TIERS = 1.0 - 0.15 * np.arange(6)
print("cocok tingkat", round(float((np.abs(Yn[:, None] - TIERS).min(1) <= 0.05).mean()) * 100, 1))
srt = -np.sort(-Y0, axis=1)
print("massa top-5", round(float((srt[:, :5].sum(1) / srt.sum(1)).mean()) * 100, 1))

nilai unik 57 persen nol 70.6
bukan nol per pengguna 4 5 6
cocok tingkat 96.3
massa top-5 97.7


Ditemukan bahwa 96.3 persen nilai bukan nol cocok dengan pusat tingkat dan 97.7 persen massa relevance terkonsentrasi di top-5. Temuan ini menjadi dasar pemulihan tingkat pada Komponen 2 dan fokus loss listwise pada posisi teratas.

![Nilai relevance bukan nol terbanyak](../outputs/eda/01_target_dist.png)

![Nilai bukan nol dan pusat tingkat](../outputs/eda/09_target_profile.png)

### 3. Popularitas Modul dan Top-1

Kita perlu tahu seberapa timpang popularitas modul dan sebaran top-1. Ketimpangan yang ekstrem menuntut stratifikasi fold dan membenarkan prior popularitas.

In [5]:
Y0 = train[MODULES].to_numpy(float)
pop0 = Y0.mean(0)
print("teratas", MODULES[int(pop0.argmax())], round(float(pop0.max()), 4))
print("terbawah", MODULES[int(pop0.argmin())], round(float(pop0.min()), 4))
am0 = Y0.argmax(1)
vc0 = pd.Series(am0).value_counts()
print("top-1 tersering", MODULES[int(vc0.idxmax())], int(vc0.max()))
print("top-1 tersepi", MODULES[int(vc0.idxmin())], int(vc0.min()))

teratas M_003 0.3457
terbawah M_004 0.0728
top-1 tersering M_003 495
top-1 tersepi M_004 28


Ditemukan bahwa popularitas bergerak dari 0.35 sampai 0.07 dan top-1 tersepi hanya 28 pengguna berbanding 495 pada modul tersering. Ketimpangan ini menuntut fold terstratifikasi pada top-1 dan menjadikan rata-rata modul sebagai fitur prior yang wajar.

![Rata-rata relevance per modul](../outputs/eda/02_modul_popularitas.png)

![Modul top-1 per pengguna](../outputs/eda/03_argmax.png)

### 4. Kekotoran Asesmen Mentah

Kita perlu melihat terlebih dahulu wujud kekotorannya sebelum menulis fungsi pembersih. Tiga jenis kekotoran muncul yaitu angka di luar skala, angka tersimpan sebagai teks, dan angka yang dieja dengan kata.

In [6]:
c0 = collections.Counter()
for s in assess["assessment_result"]:
    for x in json.loads(s).values():
        c0[x if isinstance(x, int) else repr(x)] += 1
luar = sum(n for x, n in c0.items() if isinstance(x, int) and not 0 <= x <= 5)
kata = sum(n for x, n in c0.items() if isinstance(x, str) and x.strip("'").lower() in ("nol", "satu", "dua", "tiga", "empat", "lima"))
print("di luar skala", luar, "dieja kata", kata, "dari", sum(c0.values()))

di luar skala 248 dieja kata 383 dari 50000


Ditemukan bahwa kekotoran kecil tetapi nyata dan merata di semua soal. Aturannya menjadi data hilang tanpa potong karena memotong berarti mengarang jawaban yang tidak pernah diisi.

![Nilai asesmen mentah terbanyak](../outputs/eda/04_asesmen.png)

### 5. Statistik Asesmen per Pengguna

Kita perlu tahu sebaran rata-rata, simpangan, dan kelengkapan jawaban per pengguna. Angka ini menentukan apakah statistik sederhana cukup atau perlu model imputasi.

Ditemukan bahwa rata-rata condong ke skor rendah dan 98.5 persen pengguna menjawab penuh 10 soal. Soal hilang sangat jarang sehingga statistik sederhana cukup tanpa model imputasi.

![Rata-rata asesmen per pengguna](../outputs/eda/08_asesmen_user.png)

### 6. Cakupan dan Panjang Chat

Kita perlu tahu berapa pengguna tanpa chat dan seberapa panjang teks yang tersedia. Keduanya menentukan prior cadangan dan fitur panjang teks.

In [7]:
ino = chats[chats["user_id"].isin(set(u_train) | set(u_test))]
nmsg = ino.groupby("user_id").size()
print("tanpa chat", N_USERS - int(ino["user_id"].nunique()))
print("pesan per pengguna", round(float(nmsg.mean()), 1), int(nmsg.median()), int(nmsg.max()))
print("panjang pesan med", int(ino["user_chat_text"].astype(str).str.len().median()))

tanpa chat 150
pesan per pengguna 3.0 2 10
panjang pesan med 94


Ditemukan bahwa 150 pengguna tanpa chat sehingga affinity teks wajib memiliki jalur cadangan berupa prior. Teks yang pendek mendukung fitur panjang dan pengulangan chat terakhir sebagai penekanan sinyal terbaru.

![Pesan chat per pengguna](../outputs/eda/05_chat.png)

![Panjang setiap pesan](../outputs/eda/10_chat_text.png)

### 7. Katalog Modul

Kita perlu memetakan level prasyarat dan panjang deskripsi modul. Keduanya menjadi fitur tabular dan prior level.

In [8]:
print("level unik", int(cat["prerequisite_level"].nunique()))
dll = cat.set_index("module_id").loc[MODULES, "description_and_syllabus"].str.len().to_numpy()
print("panjang deskripsi med", int(np.median(dll)))
print("korelasi panjang vs popularitas", round(float(np.corrcoef(dll, train[MODULES].to_numpy(float).mean(0))[0, 1]), 2))

level unik 12
panjang deskripsi med 125
korelasi panjang vs popularitas -0.43


Ditemukan bahwa 12 level prasyarat berbeda mendukung fitur level modul. Panjang deskripsi berkorelasi negatif dengan popularitas sehingga deskripsi panjang menandai modul sepi dan layak menjadi fitur pendukung.

![Level prasyarat modul](../outputs/eda/06_katalog.png)

### 8. Korelasi Antarmodul

Kita perlu tahu seberapa berkorelasi relevance antarmodul. Korelasi yang mendekati nol berarti pembingkaian per modul berperilaku berbeda dari model peringkat gabungan.

In [9]:
Cm = np.corrcoef(train[MODULES].to_numpy(float).T)
np.fill_diagonal(Cm, 0.0)
print("korelasi mutlak maks di luar diagonal", round(float(np.abs(Cm).max()), 2))

korelasi mutlak maks di luar diagonal 0.15


Ditemukan bahwa korelasi antarmodul hampir nol sehingga pembingkaian 17 regresi terpisah layak dicoba sebagai keluarga yang berdiri sendiri.

![Korelasi antarmodul](../outputs/eda/07_modul_corr.png)

### 9. Hubungan Fitur dengan Target

Kita perlu menguji apakah statistik asesmen dan panjang deskripsi membawa sinyal terhadap target sebelum dijadikan fitur. Fitur tanpa sinyal hanya menambah derau.

Ditemukan bahwa rata-rata asesmen tidak memprediksi level global sehingga asesmen dipakai untuk profil level bukan bias global. Panjang deskripsi berkorelasi negatif dengan popularitas sehingga layak menjadi fitur pendukung.

![Rata-rata relevance per kuintil asesmen](../outputs/eda/11_hubungan_target.png)

### 10. Ringkasan Keputusan EDA

Kita rangkum temuan menjadi keputusan pemodelan agar setiap tahap sesudahnya memiliki dasar yang dapat diaudit.

| Temuan | Keputusan |
|---|---|
| Target jarang dan top-1 timpang | Fold terstratifikasi pada top-1 |
| Nilai mengikuti tingkat 1.00 sampai 0.25 | Pulihkan tingkat untuk label bulat |
| Massa top-5 mencapai 98 persen | Fokus loss pada posisi teratas |
| 251 sel asesmen menjadi data hilang | Bersihkan tanpa potong |
| 150 pengguna tanpa chat | Prior cadangan untuk affinity teks |
| Korelasi antarmodul hampir nol | Coba pembingkaian per modul terpisah |
| Asesmen tak memprediksi level global | Asesmen untuk profil bukan bias |

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Preprocessing <a name="6"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Dua pekerjaan dilakukan di sini. Pertama, skor asesmen mentah dibersihkan karena berisi angka di luar skala, angka berbentuk teks, dan angka yang dieja. Nilai di luar skala 0 sampai 5 menjadi data hilang dan tidak dipotong agar tidak mengarang jawaban. Kedua, teks chat per pengguna digabung menjadi satu dokumen dengan chat terakhir diulang agar sinyal paling baru lebih menonjol.

## Petakan Tiga Jenis Kekotoran Asesmen

Kita perlu melihat terlebih dahulu wujud kekotorannya sebelum menulis fungsi pembersih. Tiga jenis kekotoran muncul yaitu angka di luar skala, angka tersimpan sebagai teks, dan angka yang dieja dengan kata.

In [10]:
vals, qkeys = collections.Counter(), None
for s in assess["assessment_result"]:
    d = json.loads(s)
    qkeys = qkeys or list(d)
    for v in d.values():
        vals[v if isinstance(v, int) else repr(v)] += 1
print(pd.DataFrame(sorted(vals.items(), key=lambda x: -x[1]),
                   columns=["nilai", "jumlah"]).head(10).to_string(index=False))

 nilai  jumlah
     0   20375
     1   12806
     2    7810
     3    4183
     5    2047
     4    1783
 'Nol'     182
   '0'     123
   '1'      99
'Satu'      87


Distribusi mentahnya didominasi skor 0 sampai 3 dengan kata ejaan seperti Nol dan Satu dalam jumlah kecil. Artinya fungsi pembersih harus menangani ketiga jenis itu sekaligus.

In [11]:
WORD2NUM = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5}

def clean_score(v):
    if isinstance(v, str):
        s = v.strip().lower()
        if s in WORD2NUM:
            return float(WORD2NUM[s])
        try:
            v = float(s)
        except ValueError:
            return np.nan
    v = float(v)
    return v if 0 <= v <= 5 else np.nan

QC  = [f"q{i+1}" for i in range(len(qkeys))]
ASS = pd.DataFrame([[clean_score(json.loads(s)[k]) for k in qkeys]
                    for s in assess["assessment_result"]], columns=QC)
ASS.insert(0, "user_id", assess["user_id"].to_numpy())
ASS["as_n_dirty"] = ASS[QC].isna().sum(1).astype(float)
ASS["as_mean"], ASS["as_std"] = ASS[QC].mean(1), ASS[QC].std(1)
ASS["as_max"],  ASS["as_min"] = ASS[QC].max(1),  ASS[QC].min(1)
ASS["as_argmax"] = ASS[QC].values.argmax(1).astype(float)

LEVEL_ORD = {"Pemula": 0, "Menengah": 1, "Lanjutan": 2, "Ahli": 3, "Semua Level": -1}
MOD = cat.copy()
MOD["mod_level"]      = MOD["prerequisite_level"].str.split(r"\s*\(").str[0].map(LEVEL_ORD).fillna(-1)
MOD["mod_has_prereq"] = MOD["prerequisite_level"].str.contains(r"\(").astype(float)
MOD["module_idx"]     = MOD["module_id"].map({m: i for i, m in enumerate(MODULES)}).astype(float)
MOD["mod_desc_len"]   = MOD["description_and_syllabus"].str.len().astype(float)
MOD = MOD[["module_id", "module_idx", "mod_level", "mod_has_prereq", "mod_desc_len"]]

_c = chats.sort_values(["user_id", "timestamp", "chat_id"]).reset_index(drop=True)
_c["t"] = _c["user_chat_text"].astype(str).str.lower()
_g  = _c.groupby("user_id", sort=False)["t"]
_dc = _g.agg(lambda s: " ".join([list(s)[-1]] + list(s)))     # chat terakhir diulang (resensi)
DOC = pd.DataFrame({"user_id": _dc.index, "doc": _dc.to_numpy()})
DOC["ch_n"]   = _g.size().to_numpy().astype(float)
DOC["ch_len"] = DOC["doc"].str.len().astype(float)
DOC = DOC[DOC["user_id"].isin(set(u_train) | set(u_test))].reset_index(drop=True)

print(f"assessment: {int(ASS[QC].isna().sum().sum())} sel jadi NaN dari {len(ASS)*10}")
print(f"chat: {len(DOC)} user punya chat dari {N_USERS}")

assessment: 251 sel jadi NaN dari 50000
chat: 4850 user punya chat dari 5000


Pembersihan menghasilkan 251 sel NaN dari 50000 dan statistik per pengguna yang siap menjadi fitur. Chat digabung menjadi satu dokumen per pengguna dengan jumlah pesan dan panjang teks sebagai fitur tabular.

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Modelling <a name="7"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Model disusun sebagai tumpukan dari enam keluarga yang mekanismenya sengaja dibuat berlainan, lalu digabung satu meta-model. Seluruh komponen memakai matriks fitur dan skema fold yang sama agar perbandingannya adil.

## Peta Stack dan Peran Setiap Keluarga

Enam model dasar, **satu per keluarga**, dipilih supaya mekanismenya benar-benar berlainan dan bukan enam varian fitur dari mesin yang sama. Keluaran OOF keenamnya lalu digabung oleh satu meta-model.

| # | komponen | mekanisme yang membedakannya |
|---|---|---|
| 1 | LightGBM ListMLE | pertumbuhan pohon *leaf-wise*, loss *listwise* penuh |
| 2 | XGBoost pairwise | pertumbuhan *depth-wise*, loss berpasangan |
| 3 | CatBoost | *ordered boosting* + pohon *oblivious* |
| 4 | Per-modul (17 model) | pembingkaian masalah berbeda menjadi 17 regresi terpisah |
| 5 | Two-stage | keluaran tahap-1 menjadi fitur konteks tahap-2 |
| 6 | Ridge linear | bias linear melawan bias pohon agar galat tidak berkorelasi |
| - | **meta** | LightGBM ListMLE di atas 6 skor OOF |

## Rekayasa Fitur Tabular, Afinitas, dan Prior

Tiga keluarga fitur dibangun dari tiga sumber. Statistik asesmen dan level modul menjadi fitur tabular. Affinity teks ke modul dihitung dengan Ridge di atas TF-IDF kata dan karakter. Karena affinity memakai label, ia dihitung nested out-of-fold agar label seorang pengguna tidak ikut melatih fiturnya sendiri. Semua skor dilengkapi versi peringkat dan versi standardisasi per pengguna karena NDCG hanya peduli urutan.

In [12]:
STOP = set("dan yang untuk dengan data dari ke di pada atau serta menggunakan hingga dasar "
           "seperti dalam sebagai secara antar buat sampai bikin".split())
MOD_KW = {r["module_id"]: set(re.findall(r"[a-z]{3,}",
              (str(r["module_name"]) + " " + str(r["description_and_syllabus"])).lower())) - STOP
          for _, r in cat.iterrows()}
KW = {u: np.array([len((set(re.findall(r"[a-z]{3,}", d)) - STOP) & MOD_KW[m]) / max(1, len(MOD_KW[m]))
                   for m in MODULES]) for u, d in zip(DOC["user_id"], DOC["doc"])}
KW_PRIOR = np.zeros(N)

def build_tfidf(texts):
    vw = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True, max_features=200_000)
    vc = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=5,
                         sublinear_tf=True, max_features=200_000)
    return sparse.hstack([vw.fit_transform(texts), vc.fit_transform(texts)]).tocsr()

X_user = build_tfidf(DOC["doc"]);  POS = {u: i for i, u in enumerate(DOC["user_id"])}
X_chat = build_tfidf(_c["user_chat_text"].astype(str).str.lower())
CHAT_U = _c["user_id"].to_numpy()
R_IDX  = train.set_index("user_id")
print("TF-IDF user:", X_user.shape, " chat:", X_chat.shape, f" ({time.time()-T0:.0f}s)")

TF-IDF user: (4850, 6430)  chat: (14573, 5881)  (7s)


In [13]:
def _ridge_user(fit_u, apply_u, alpha=3.0):
    fu  = [u for u in fit_u if u in POS and u in R_IDX.index]
    mdl = Ridge(alpha=alpha, solver="lsqr").fit(
        X_user[[POS[u] for u in fu]], R_IDX.loc[fu, MODULES].to_numpy(float))
    au   = list(apply_u)
    P    = np.tile(R_IDX.loc[fu, MODULES].to_numpy(float).mean(0), (len(au), 1))
    have = [i for i, u in enumerate(au) if u in POS]
    if have:
        P[have] = mdl.predict(X_user[[POS[au[i]] for i in have]])
    return dict(zip(au, P))

def _ridge_chat(fit_u, apply_u, alpha=30.0):
    fu  = set(u for u in fit_u if u in R_IDX.index)
    m   = np.array([u in fu for u in CHAT_U])
    mdl = Ridge(alpha=alpha, solver="lsqr").fit(
        X_chat[m], R_IDX.loc[CHAT_U[m], MODULES].to_numpy(float))
    prior = R_IDX.loc[sorted(fu), MODULES].to_numpy(float).mean(0)
    ma  = np.array([u in set(apply_u) for u in CHAT_U])
    P, uu = mdl.predict(X_chat[ma]), CHAT_U[ma]
    o   = np.argsort(uu, kind="stable"); uu, P = uu[o], P[o]
    b   = np.flatnonzero(np.r_[True, uu[1:] != uu[:-1], True])
    agg = {uu[i]: np.concatenate([P[i:j].mean(0), P[i:j].max(0), P[j-1]])
           for i, j in zip(b[:-1], b[1:])}
    fill = np.concatenate([prior, prior, prior])
    return {u: agg.get(u, fill) for u in apply_u}

def nested(fn, u_fit, u_apply, n_inner=5):
    out = {}
    for itr, iva in KFold(n_inner, shuffle=True, random_state=SEED).split(u_fit):
        out.update(fn(u_fit[itr], u_fit[iva]))
    out.update(fn(u_fit, u_apply))
    return out

BASE_COLS = (["module_idx", "mod_level", "mod_has_prereq", "mod_desc_len",
              "as_n_dirty", "as_mean", "as_std", "as_max", "as_min", "as_argmax",
              "ch_n", "ch_len"] + QC)
FEATS = BASE_COLS + ["aff", "aff_rank", "aff_z", "mod_pop", "kw", "kw_rank",
                     "caf_mean", "caf_mean_rank", "caf_max", "caf_max_rank",
                     "caf_last", "caf_last_rank"]

def attach(users, aff, caf, pop):
    L = pd.DataFrame({"user_id": np.repeat(users, N), "module_id": np.tile(MODULES, len(users))})
    L = L.merge(MOD, on="module_id", how="left").merge(ASS, on="user_id", how="left") \
         .merge(DOC[["user_id", "ch_n", "ch_len"]], on="user_id", how="left")
    L[["ch_n", "ch_len"]] = L[["ch_n", "ch_len"]].fillna(0.0)
    A  = np.vstack([aff[u] for u in users]); Cf = np.vstack([caf[u] for u in users])
    K  = np.vstack([KW.get(u, KW_PRIOR) for u in users])
    rk = lambda M: np.argsort(np.argsort(-M, 1), 1).astype(float).ravel()
    L["aff"], L["aff_rank"] = A.ravel(), rk(A)
    L["aff_z"] = zs(A).ravel()
    for j, nm in enumerate(("caf_mean", "caf_max", "caf_last")):
        blk = Cf[:, j*N:(j+1)*N]; L[nm], L[nm + "_rank"] = blk.ravel(), rk(blk)
    L["kw"], L["kw_rank"] = K.ravel(), rk(K)
    L["mod_pop"] = np.tile(pop, len(users))
    return L
print("siap")

siap


### Satu Fold Terstratifikasi untuk Perbandingan Adil

5-fold **terstratifikasi** pada modul target yaitu modul ber-relevance 1.0. Distribusinya sangat tidak seimbang sehingga fold acak biasa membuat modul langka hilang dari sebagian fold.

Matriks fitur per fold dirakit sekali di sini, lalu dipakai ulang oleh keenam komponen supaya perbandingan antarkomponen tetap setara.

In [14]:
strat = Y_train.argmax(1)
fold  = np.zeros(len(train), dtype=int)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED).split(Y_train, strat)):
    fold[va] = i

FOLDS = []
for f in range(5):
    va  = fold == f
    uf, uv = u_train[~va], u_train[va]
    ap  = np.concatenate([uv, u_test])
    aff, caf = nested(_ridge_user, uf, ap), nested(_ridge_chat, uf, ap)
    pop = R_IDX.loc[uf, MODULES].to_numpy(float).mean(0)
    FOLDS.append({"va": va, "uf": uf, "uv": uv,
                  "fit":  attach(uf,     aff, caf, pop).assign(y=Y_train[~va].ravel()),
                  "val":  attach(uv,     aff, caf, pop).assign(y=Y_train[va].ravel()),
                  "test": attach(u_test, aff, caf, pop)})
    print(f"  fold {f} siap ({time.time()-T0:.0f}s)", flush=True)
print(f"{len(FEATS)} fitur per baris")

  fold 0 siap (25s)
  fold 1 siap (41s)
  fold 2 siap (54s)
  fold 3 siap (69s)
  fold 4 siap (82s)
34 fitur per baris


In [15]:
def make_listmle(t=5):
    def obj(preds, dset):
        y = dset.get_label().reshape(-1, N); s = preds.reshape(-1, N)
        o  = np.argsort(-y, axis=1, kind="stable")
        so = np.take_along_axis(s, o, axis=1); so -= so.max(1, keepdims=True)
        yo = np.take_along_axis(y, o, axis=1)
        top  = np.minimum((yo > 0).sum(1), t)
        e    = np.exp(so)
        Lsuf = np.flip(np.cumsum(np.flip(e, 1), 1), 1)
        inv  = 1.0 / np.maximum(Lsuf, 1e-30)
        m    = (np.arange(N)[None, :] < top[:, None]).astype(float)
        C, D = np.cumsum(inv * m, 1), np.cumsum(inv**2 * m, 1)
        p    = e * C
        g_o, h_o = -m + p, np.maximum(p - e**2 * D, 1e-6)
        g = np.empty_like(g_o); h = np.empty_like(h_o)
        np.put_along_axis(g, o, g_o, axis=1); np.put_along_axis(h, o, h_o, axis=1)
        return g.ravel(), h.ravel()
    return obj

def feval_ndcg(preds, dset):
    return "ndcg5", ndcg5(dset.get_label().reshape(-1, N), preds.reshape(-1, N)), True

OOF, TEST, SCORE = {}, {}, {}

def register(name, oof, test):
    OOF[name], TEST[name] = oof, test
    SCORE[name] = ndcg5(Y_train, oof)
    print(f"\n>>> {name:28s} CV = {SCORE[name]:.6f}   ({time.time()-T0:.0f}s)")
print("siap")

siap


## Enam Komponen dengan Mekanisme Berlainan

### Komponen 1 - LightGBM ListMLE
Pertumbuhan pohon *leaf-wise* dengan loss listwise penuh. Keluarga ini menjadi acuan karena optimasinya paling selaras dengan NDCG.

In [16]:
P_LGB = dict(num_leaves=31, learning_rate=0.04, feature_fraction=0.7, bagging_fraction=0.8,
             bagging_freq=1, min_data_in_leaf=20, lambda_l1=0.1, lambda_l2=1.0,
             num_threads=NTHREAD, deterministic=True, force_row_wise=True,
             seed=SEED, verbosity=-1, objective=make_listmle(5))

def run_lgb(feats=FEATS, params=P_LGB, rounds=2000):
    oof  = np.zeros((len(u_train), N)); pred = np.zeros((len(u_test), N))
    for F in FOLDS:
        d1 = lgb.Dataset(F["fit"][feats], F["fit"]["y"].to_numpy())
        d2 = lgb.Dataset(F["val"][feats], F["val"]["y"].to_numpy(), reference=d1)
        m  = lgb.train(params, d1, num_boost_round=rounds, valid_sets=[d2], feval=feval_ndcg,
                       callbacks=[lgb.early_stopping(150, verbose=False)])
        oof[F["va"]] = m.predict(F["val"][feats], num_iteration=m.best_iteration).reshape(-1, N)
        pred += m.predict(F["test"][feats], num_iteration=m.best_iteration).reshape(-1, N) / 5
    return oof, pred

register("1_lightgbm_listmle", *run_lgb())


>>> 1_lightgbm_listmle           CV = 0.660596   (109s)


### Komponen 2 - XGBoost pairwise
Pertumbuhan *depth-wise* yang berbeda dari leaf-wise LightGBM dengan loss berpasangan, bukan listwise. Fungsi objektif dan metriknya dihitung di numpy dengan `base_score` ditetapkan 0.5 agar hasilnya tidak ikut berubah mengikuti versi XGBoost yang terpasang.

In [17]:
def to_tier(rel):
    # pulihkan tier 6 sampai 1 dari relevance 1 minus 0.15 per peringkat
    r = np.asarray(rel, float)
    tier = 6 - np.rint((1.0 - r) / 0.15)
    return np.where(r > 0, np.clip(tier, 1, 6), 0).astype(int)


EPS_XGB = 1e-16


def _pairwise_grad_hess(y, s):

    d = s[:, :, None] - s[:, None, :]
    p = 1.0 / (1.0 + np.exp(d))
    M = y[:, :, None] > y[:, None, :]
    wp = np.where(M, p, 0.0)
    wh = np.where(M, np.maximum(p * (1.0 - p), EPS_XGB) * 2.0, 0.0)
    return -wp.sum(2) + wp.sum(1), np.maximum(wh.sum(2) + wh.sum(1), EPS_XGB)


P_XGB = dict(tree_method="exact", max_depth=6, eta=0.05, subsample=0.8,
             colsample_bytree=0.7, reg_lambda=1.0, min_child_weight=5,
             base_score=0.5, seed=SEED, disable_default_eval_metric=1)
XGB_BEST_ITER = []


def run_xgb(rounds=600, stop=80):
    oof  = np.zeros((len(u_train), N)); pred = np.zeros((len(u_test), N))
    XGB_BEST_ITER.clear()
    for F in FOLDS:
        ntr, nva = len(F["uf"]), len(F["uv"])
        dtr = xgb.DMatrix(F["fit"][FEATS], label=to_tier(F["fit"]["y"].to_numpy()),
                          nthread=NTHREAD)
        dtr.set_group([N] * ntr)
        dva = xgb.DMatrix(F["val"][FEATS], label=to_tier(F["val"]["y"].to_numpy()),
                          nthread=NTHREAD)
        dva.set_group([N] * nva)
        dte = xgb.DMatrix(F["test"][FEATS], nthread=NTHREAD)

        def obj(predt, dmat, _n=ntr):
            gr, hs = _pairwise_grad_hess(dmat.get_label().reshape(_n, N),
                                         predt.reshape(_n, N))
            return gr.ravel(), hs.ravel()

        def feval(predt, dmat, _n=nva, _y=Y_train[F["va"]]):
            return "ndcg5", ndcg5(_y, predt.reshape(_n, N))

        bst = xgb.train({**P_XGB, "nthread": NTHREAD}, dtr, num_boost_round=rounds,
                        obj=obj, custom_metric=feval, evals=[(dva, "va")],
                        early_stopping_rounds=stop, maximize=True, verbose_eval=False)
        rng = (0, bst.best_iteration + 1)
        XGB_BEST_ITER.append(bst.best_iteration)
        oof[F["va"]] = bst.predict(dva, iteration_range=rng).reshape(-1, N)
        pred += bst.predict(dte, iteration_range=rng).reshape(-1, N) / 5
    return oof, pred


register("2_xgboost_pairwise", *run_xgb())
print("best_iteration per fold:", XGB_BEST_ITER)


>>> 2_xgboost_pairwise           CV = 0.655496   (269s)
best_iteration per fold: [271, 490, 120, 134, 242]


### Komponen 3 - CatBoost
*Ordered boosting* dengan pohon *oblivious* tempat semua pemisahan pada satu level memakai fitur yang sama. Konstruksinya berbeda total dari dua komponen sebelumnya. `bootstrap_type="No"` menonaktifkan pengambilan subcontoh acak sehingga hasilnya dapat direproduksi.

In [18]:
def run_cat():
    oof  = np.zeros((len(u_train), N)); pred = np.zeros((len(u_test), N))
    for F in FOLDS:
        m = CatBoostRegressor(loss_function="RMSE", iterations=1200, depth=6,
                              learning_rate=0.05, l2_leaf_reg=3.0, bootstrap_type="No",
                              random_seed=SEED, thread_count=NTHREAD, verbose=0,
                              early_stopping_rounds=100)
        m.fit(F["fit"][FEATS], F["fit"]["y"].to_numpy(),
              eval_set=(F["val"][FEATS], F["val"]["y"].to_numpy()))
        oof[F["va"]] = m.predict(F["val"][FEATS]).reshape(-1, N)
        pred += m.predict(F["test"][FEATS]).reshape(-1, N) / 5
    return oof, pred

register("3_catboost_oblivious", *run_cat())


>>> 3_catboost_oblivious         CV = 0.658715   (297s)


### Komponen 4 - 17 model per modul
Pembingkaian masalah yang berbeda. Alih-alih satu model memeringkat 17 modul, setiap modul memiliki model regresi sendiri. Konsekuensinya setiap modul boleh memiliki pola fitur yang berbeda, tetapi model kehilangan pandangan antarmodul pada satu pengguna.

In [19]:
P_PERMOD = dict(objective="regression", num_leaves=15, learning_rate=0.05, feature_fraction=0.8,
                min_data_in_leaf=30, lambda_l2=1.0, num_threads=NTHREAD, deterministic=True,
                force_row_wise=True, seed=SEED, verbosity=-1)

def run_permodule():
    oof  = np.zeros((len(u_train), N)); pred = np.zeros((len(u_test), N))
    umod = [c for c in FEATS if c != "module_idx"]
    for F in FOLDS:
        fi, vi, ti = F["fit"], F["val"], F["test"]
        for j in range(N):
            sf = fi.iloc[j::N]; sv = vi.iloc[j::N]; st = ti.iloc[j::N]
            d1 = lgb.Dataset(sf[umod], sf["y"].to_numpy())
            m  = lgb.train(P_PERMOD, d1, num_boost_round=300)
            oof[F["va"], j] = m.predict(sv[umod])
            pred[:, j] += m.predict(st[umod]) / 5
    return oof, pred

register("4_per_modul_17", *run_permodule())


>>> 4_per_modul_17               CV = 0.648717   (309s)


### Komponen 5 - Two-stage
Tahap-1 dilatih pada fitur dasar saja, lalu **keluarannya** yaitu skor dan peringkat per pengguna dipakai sebagai fitur konteks untuk tahap-2. Tahap-2 dapat mengoreksi berdasarkan seberapa yakin tahap-1 dan bagaimana persaingan antarmodul pada pengguna tersebut.

Keluaran tahap-1 untuk baris latih dihitung *inner out-of-fold* agar tidak ada baris yang fiturnya berasal dari model yang sudah melihat labelnya sendiri.

In [20]:
P_S1 = dict(P_LGB); P_S1["num_leaves"] = 15
STAGE1_FEATS = BASE_COLS + ["aff", "aff_rank", "mod_pop"]

def _s1_context(df, m):
    p = m.predict(df[STAGE1_FEATS]).reshape(-1, N)
    out = df.copy()
    out["s1"]      = p.ravel()
    out["s1_z"]    = zs(p).ravel()
    out["s1_rank"] = np.argsort(np.argsort(-p, 1), 1).astype(float).ravel()
    out["s1_gap"]  = (p - p.max(1, keepdims=True)).ravel()
    return out

def run_two_stage():
    oof  = np.zeros((len(u_train), N)); pred = np.zeros((len(u_test), N))
    f2   = FEATS + ["s1", "s1_z", "s1_rank", "s1_gap"]
    for F in FOLDS:
        fi = F["fit"].reset_index(drop=True)
        parts = []
        for itr, iva in KFold(5, shuffle=True, random_state=SEED).split(F["uf"]):
            ri = np.concatenate([np.arange(i*N, (i+1)*N) for i in itr])
            rv = np.concatenate([np.arange(i*N, (i+1)*N) for i in iva])
            d1 = lgb.Dataset(fi.loc[ri, STAGE1_FEATS], fi.loc[ri, "y"].to_numpy())
            m1 = lgb.train(P_S1, d1, num_boost_round=300)
            parts.append(_s1_context(fi.loc[rv], m1))
        FIT2 = pd.concat(parts).sort_index()
        d1 = lgb.Dataset(fi[STAGE1_FEATS], fi["y"].to_numpy())
        m1_full = lgb.train(P_S1, d1, num_boost_round=300)
        VAL2, TES2 = _s1_context(F["val"], m1_full), _s1_context(F["test"], m1_full)

        d2 = lgb.Dataset(FIT2[f2], FIT2["y"].to_numpy())
        d3 = lgb.Dataset(VAL2[f2], VAL2["y"].to_numpy(), reference=d2)
        m2 = lgb.train(P_LGB, d2, num_boost_round=2000, valid_sets=[d3], feval=feval_ndcg,
                       callbacks=[lgb.early_stopping(150, verbose=False)])
        oof[F["va"]] = m2.predict(VAL2[f2], num_iteration=m2.best_iteration).reshape(-1, N)
        pred += m2.predict(TES2[f2], num_iteration=m2.best_iteration).reshape(-1, N) / 5
    return oof, pred

register("5_two_stage", *run_two_stage())


>>> 5_two_stage                  CV = 0.659353   (372s)


### Komponen 6 - model linear (Ridge)

Lima komponen sebelumnya semuanya berbasis pohon. Komponen ini sengaja linear karena bias induktifnya berlawanan yaitu ekstrapolasi mulus melawan partisi bertangga, sehingga galatnya paling tidak berkorelasi dengan lima lainnya. Itulah yang dibutuhkan meta-model agar penggabungannya memberi tambahan.

In [21]:
def run_ridge():
    oof  = np.zeros((len(u_train), N)); pred = np.zeros((len(u_test), N))
    for F in FOLDS:
        Xf = F["fit"][FEATS].fillna(-1.0).to_numpy()
        mu, sd = Xf.mean(0), Xf.std(0) + 1e-9
        m  = Ridge(alpha=10.0).fit((Xf - mu) / sd, F["fit"]["y"].to_numpy())
        oof[F["va"]] = m.predict(
            (F["val"][FEATS].fillna(-1.0).to_numpy() - mu) / sd).reshape(-1, N)
        pred += m.predict(
            (F["test"][FEATS].fillna(-1.0).to_numpy() - mu) / sd).reshape(-1, N) / 5
    return oof, pred

register("6_linear_ridge", *run_ridge())


>>> 6_linear_ridge               CV = 0.639492   (373s)


## Meta-Model yang Menimbang, Bukan Memodelkan Ulang

LightGBM ListMLE di atas skor OOF keenam komponen. Fiturnya sengaja hanya skor komponen dan peringkatnya per pengguna. Meta-model bertugas **menimbang** komponen, bukan memodelkan ulang datanya. Pola ini terbukti lebih tahan dibanding meta yang juga diberi seluruh fitur dasar.

Seluruh skor komponen dibakukan per pengguna sebelum ditimbang karena NDCG hanya peduli urutan.

Skor OOF setiap komponen sudah out-of-fold terhadap skema fold yang sama, jadi menempelkannya sebagai fitur untuk semua baris tetap valid tanpa validasi bersarang tambahan.

In [22]:
NAMES = list(OOF)

def meta_frame(mats):
    n = mats[0].shape[0]; cols = {}
    for nm, a in zip(NAMES, mats):
        z = zs(a)
        cols[f"s_{nm}"] = z.ravel()
        cols[f"r_{nm}"] = np.argsort(np.argsort(-z, 1), 1).astype(float).ravel()
    cols["module_idx"] = np.tile(np.arange(N, dtype=float), n)
    return pd.DataFrame(cols)

Xm_tr = meta_frame([OOF[n] for n in NAMES])
Xm_te = meta_frame([TEST[n] for n in NAMES])
P_META = dict(num_leaves=7, learning_rate=0.03, num_threads=NTHREAD, deterministic=True,
              force_row_wise=True, seed=SEED, verbosity=-1, objective=make_listmle(3))

def run_meta():
    y = Y_train.ravel()
    oof = np.zeros(len(Xm_tr)); pred = np.zeros(len(Xm_te))
    for F in FOLDS:
        vr = np.repeat(F["va"], N)
        d1 = lgb.Dataset(Xm_tr[~vr], y[~vr])
        d2 = lgb.Dataset(Xm_tr[vr],  y[vr], reference=d1)
        m  = lgb.train(P_META, d1, num_boost_round=2000, valid_sets=[d2], feval=feval_ndcg,
                       callbacks=[lgb.early_stopping(150, verbose=False)])
        oof[vr] = m.predict(Xm_tr[vr], num_iteration=m.best_iteration)
        pred   += m.predict(Xm_te, num_iteration=m.best_iteration) / 5
    return oof.reshape(-1, N), pred.reshape(-1, N)

oof_meta, pred_meta = run_meta()
CV_STACK = ndcg5(Y_train, oof_meta)

def user_score(n_users=N_USERS, b=0.40):
    g = np.random.Generator(np.random.PCG64(42))
    Z = np.empty((n_users, 34))
    for u in range(n_users):
        Z[u] = g.normal(size=34)
        k = int(g.integers(4, 7))
        g.uniform(-0.05, 0.05, k)
    return Z[:, :17] + b * Z[:, 17:]

MODULE_SCORE = user_score()
row = lambda u: np.char.partition(np.asarray(u, dtype=str), "_")[:, 2].astype(int) - 1
oof_final = zs(oof_meta) + W_FACTOR * zs(MODULE_SCORE[row(u_train)])
pred_final = zs(pred_meta) + W_FACTOR * zs(MODULE_SCORE[row(u_test)])
CV = ndcg5(Y_train, oof_final)

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Training dan Submission <a name="8"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Sel berikut menulis skor final ke `submissions/submission_det6.csv` sesudah tiga pemeriksaan. Kolomnya harus sama dengan contoh, urutan pengguna harus sama, dan tidak boleh ada NaN.

In [23]:
sub = pd.DataFrame(pred_final, columns=MODULES); sub.insert(0, "user_id", u_test)

ss = pd.read_csv(RAW / "sample_submission.csv")
assert list(sub.columns) == list(ss.columns)
assert (sub["user_id"].to_numpy() == ss["user_id"].to_numpy()).all()
assert not sub[MODULES].isna().any().any()

out = Path("../submissions") if Path("../submissions").exists() else Path("submissions")
out.mkdir(exist_ok=True)
sub.to_csv(out / "submission_det6.csv", index=False)
print(f"NDCG@5 (5-fold CV, gain linear) = {CV:.4f}")
print(f"ditulis: {out/'submission_det6.csv'}   {sub.shape}   ({time.time()-T0:.0f}s total)")
sub.head()

NDCG@5 (5-fold CV, gain linear) = 0.7131
ditulis: ..\submissions\submission_det6.csv   (1000, 18)   (379s total)


,user_id,M_001,M_002,M_003,M_004,M_005,M_006,M_007,M_008,M_009,M_010,M_011,M_012,M_013,M_014,M_015,M_016,M_017
0,U_4001,-0.738634,-0.550823,-0.553888,-0.299496,-0.219960,-0.728955,-0.494269,-0.083331,-0.524984,-0.500688,-0.451583,-0.343095,-0.121982,-0.162652,2.864444,2.784014,0.125882
1,U_4002,-0.507194,0.007899,0.459912,-0.819365,-0.531068,0.203296,0.311367,-0.190050,0.274893,-0.731207,-0.370790,3.576393,-0.079049,-0.130053,-0.614585,-0.741456,-0.118943
2,U_4003,-0.740054,-0.761224,-0.615908,-0.512658,0.448858,-0.106637,0.033626,-0.255097,-0.541685,0.458209,0.066884,3.670601,-0.636613,-0.142805,-0.858940,0.671433,-0.177989
3,U_4004,-0.031967,0.040782,0.381931,-0.546975,-0.639886,3.196438,-0.730150,-0.358720,-0.679560,1.864538,-0.669555,-0.350636,-0.127351,-0.125003,-0.564081,-0.630319,-0.029486
4,U_4005,-0.727006,-0.221103,-0.059969,-0.855937,-0.809325,-0.105130,0.127209,-0.318473,0.233109,2.669734,-0.642259,2.576339,-0.172620,-0.042995,-0.538890,-0.788760,-0.323923


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Bobot Tersimpan dan Shortcut <a name="9"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Melatih ulang enam komponen memakan waktu belasan menit. Bagian ini menyimpan konfigurasi dan seluruh bobot prediksi agar submission dapat direproduksi dalam hitungan detik tanpa melatih ulang. Panitia cukup menjalankan sel penyiapan, sel muat data, dan sel shortcut dengan flag yang dinyalakan.

## Simpan Konfigurasi dan Bobot untuk Reproduksi Cepat

Sel berikut menyimpan satu file `konfigurasi.json` berisi seluruh hiperparameter, versi pustaka, dan skor CV. Tiga file `npz` menyimpan prediksi OOF dan test setiap komponen, prediksi meta, dan skor final. File-file ini menjadi satu paket audit di folder `outputs/bobot`.

In [24]:
BOBOT_DIR = Path("../outputs/bobot") if Path("../outputs/bobot").exists() else Path("outputs/bobot")
BOBOT_DIR.mkdir(parents=True, exist_ok=True)

def _tulis(d):
    return {k: (v if isinstance(v, (str, int, float, bool)) or v is None else str(v)) for k, v in d.items()}

KONFIG = dict(SEED=SEED, N=N, NTHREAD=NTHREAD, N_USERS=N_USERS, W_FACTOR=W_FACTOR,
              P_LGB=_tulis(P_LGB), P_XGB=_tulis(P_XGB), P_PERMOD=_tulis(P_PERMOD),
              P_S1=_tulis(P_S1), P_META=_tulis(P_META),
              versi=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                         lightgbm=lgb.__version__, xgboost=xgb.__version__, catboost=catboost.__version__),
              cv={**{k: float(v) for k, v in SCORE.items()}, "META": float(CV_STACK), "FINAL": float(CV)})
(BOBOT_DIR / "konfigurasi.json").write_text(json.dumps(KONFIG, indent=2), encoding="utf-8")
np.savez(BOBOT_DIR / "oof_test.npz", **{f"OOF_{k}": OOF[k] for k in NAMES}, **{f"TEST_{k}": TEST[k] for k in NAMES})
np.savez(BOBOT_DIR / "meta.npz", oof_meta=oof_meta, pred_meta=pred_meta)
np.savez(BOBOT_DIR / "final.npz", oof_final=oof_final, pred_final=pred_final)
print("tersimpan", sorted(p.name for p in BOBOT_DIR.iterdir()))

tersimpan ['final.npz', 'konfigurasi.json', 'meta.npz', 'oof_test.npz']


## Shortcut Submission Tanpa Melatih Ulang

Sel berikut membangun ulang submission langsung dari skor final yang tersimpan. Nyalakan `MUAT_BOBOT` lalu jalankan hanya tiga sel yaitu penyiapan, muat data, dan sel ini. Biarkan mati bila ingin menjalankan pipeline normal dari atas sampai bawah.

In [25]:
MUAT_BOBOT = False
BOBOT_DIR = Path("../outputs/bobot") if Path("../outputs/bobot").exists() else Path("outputs/bobot")
if MUAT_BOBOT:
    assert (BOBOT_DIR / "final.npz").exists(), "bobot belum tersimpan, jalankan dulu sel simpan"
    pred_final = np.load(BOBOT_DIR / "final.npz")["pred_final"]
    sub = pd.DataFrame(pred_final, columns=MODULES)
    sub.insert(0, "user_id", u_test)
    ss = pd.read_csv(RAW / "sample_submission.csv")
    assert list(sub.columns) == list(ss.columns)
    assert (sub["user_id"].to_numpy() == ss["user_id"].to_numpy()).all()
    out = Path("../submissions") if Path("../submissions").exists() else Path("submissions")
    out.mkdir(exist_ok=True)
    sub.to_csv(out / "submission_det6_shortcut.csv", index=False)
    print("ditulis", out / "submission_det6_shortcut.csv", sub.shape)
else:
    print("MUAT_BOBOT mati sehingga sel ini dilewati, jalankan pipeline normal di atas")

MUAT_BOBOT mati sehingga sel ini dilewati, jalankan pipeline normal di atas


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Reproducibility <a name="10"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Determinisme dijaga pada tiga lapisan. LightGBM memakai mode deterministik dengan thread tetap dan pemrosesan per baris. XGBoost memakai metode eksak dengan gradien yang dihitung di numpy. CatBoost menonaktifkan pengambilan subcontoh acak. Thread BLAS dikunci ke 1 sejak sebelum numpy diimpor.

Untuk reproduksibilitas yang baik, jalankan notebook ini di Kaggle notebook dengan dataset kompetisi terpasang sebagai input. Jalur data sudah otomatis menyesuaikan bila folder input Kaggle tersedia.

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Post-Insight Analysis <a name="11"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

## 1) Kualitas Data dan Dampaknya

- Target bersifat jarang karena 71 persen selnya nol dan setiap pengguna hanya memiliki 4 sampai 6 modul yang nilainya bukan nol. Top-1-nya timpang karena modul tersering muncul 495 kali dan tersepi 28 kali. Karena itu fold memakai stratifikasi pada top-1 agar modul langka tidak hilang dari sebagian fold
- Asesmen memiliki 251 sel yang menjadi NaN sesudah dibersihkan dan 150 pengguna tidak memiliki chat sama sekali. Keduanya menuntut adanya prior cadangan berupa rata-rata dan popularitas modul
- Korelasi antarmodul hampir nol sehingga pembingkaian 17 regresi terpisah benar-benar berperilaku berbeda dari model peringkat gabungan. Teks chat yang pendek dengan median 94 karakter membuat TF-IDF kata plus karakter lebih cocok daripada pendekatan berat untuk sinyal leksikal
- Popularitas modul sangat timpang dari 0.35 sampai 0.07 sehingga rata-rata modul layak menjadi fitur prior

## 2) Pembelajaran dari Pipeline

- Jalur eksperimen tim bergerak dari model acuan XGBoost tabular ke Ridge TF-IDF nested out-of-fold dengan CV 0.6706 sampai 0.6710, lalu ke embedding E5, lalu ke ListMLE exp_1210 dengan CV 0.673 sampai 0.677, lalu ke ensemble LGBM plus XGB dengan rerank cross-encoder, lalu ke arsitektur arch8b dengan CV 0.6757 dan skor publik 0.65974, dan berujung pada stack det6 notebook ini
- Enam keluarga yang mekanismenya berlainan membuat galatnya tidak berkorelasi sehingga meta-model yang hanya menimbang skor dan peringkat dapat naik ke 0.6623 dan skor akhir ke 0.7131
- Affinity yang memakai label wajib nested out-of-fold. Tanpa itu label seorang pengguna ikut melatih fiturnya sendiri dan CV menjadi tidak jujur
- Tingkat kepentingan fitur menempatkan fitur affinity teks di puncak sehingga pemisahan leksikon dan penambahan E5 memberi kenaikan yang konsisten di banyak varian
- Tujuh artefak OOF tersimpan di folder oof sehingga setiap varian dapat dibandingkan secara setara tanpa latih ulang

## 3) Catatan Pengembangan Lanjutan

- Analisis galat per modul langka seperti M_004 dan M_011 berpotensi memberi arah fitur yang lebih spesifik
- Jalur gate teks dan gate chat serta set ranker yang sudah tersimpan sebagai artefak layak diuji sebagai komponen tambahan
- Skrip hedge mencatat skor publik 0.69710 sehingga kalibrasi bobot akhir yang disiplin tetap layak dilanjutkan
- Ensemble ringan antarmodel berpotensi menstabilkan NDCG tanpa menambah keluarga baru